# Medical RAG Assistant Notebook

This notebook builds and serves a Retrieval-Augmented Generation (RAG) assistant for medical Q&A using a Hugging Face systhetic medical dialogue dataset and LangChain.

## What this notebook covers
- Loads medical dialogue data from Hugging Face.
- Converts rows into LangChain `Document` objects and chunks them.
- Creates/reuses a persisted Chroma vector store for retrieval.
- Answers user questions using retrieved context + an LLM prompt.
- Launches a Gradio chat interface directly from the notebook.

## Typical run order
1. Run setup and environment cells (API key, model, embeddings).
2. Run ingestion/vector-store helpers.
3. Run answer/retrieval helpers.
4. Launch the Gradio UI cell and chat with the assistant.

## Notes
- Ingestion is conditional and skips reloading when vectors already exist.
- Set `force_reingest=True` when you want to rebuild the vector store.

In [ ]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
DB_NAME = "medical_rag_v1"

load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY is not set in the environment variables")


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=api_key, base_url="https://openrouter.ai/api/v1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
def load_documents_from_hf(repo_id: str) -> list[Document]:
    """
    Load a Hugging Face dataset split and convert rows into LangChain Documents.

    Each document stores the row dialogue as `page_content` and keeps source
    metadata (`source_id`, `niche`) for traceable retrieval and citations.

    Args:
        repo_id (str): Hugging Face dataset repository ID.

    Returns:
        list[Document]: Unchunked LangChain documents created from dataset rows.

    Raises:
        ValueError: If `repo_id` is empty or dataset rows are missing required fields.
        RuntimeError: If the dataset cannot be loaded.
    """
    if not isinstance(repo_id, str) or not repo_id.strip():
        raise ValueError("`repo_id` must be a non-empty string.")

    print(f"📥 Loading dataset: {repo_id}")
    try:
        dataset = load_dataset(repo_id, split="train")
    except Exception as exc:
        raise RuntimeError(f"Failed to load dataset from Hugging Face: {repo_id}") from exc

    docs = []
    for i, entry in enumerate(dataset):
        if "dialogue" not in entry or "niche" not in entry:
            raise ValueError(
                "Dataset rows must include both 'dialogue' and 'niche' fields. "
                f"Invalid row index: {i}."
            )

        metadata = {"source_id": i, "niche": entry["niche"]}
        docs.append(Document(page_content=entry["dialogue"], metadata=metadata))

    if not docs:
        raise ValueError("No documents were created from the dataset.")

    return docs


def chunk_documents(
    documents: list[Document],
    chunk_size: int = 600,
    chunk_overlap: int = 100,
) -> list[Document]:
    """
    Split LangChain documents into smaller overlapping chunks for retrieval.

    Args:
        documents (list[Document]): Source documents to split.
        chunk_size (int, optional): Maximum size for each chunk. Defaults to 600.
        chunk_overlap (int, optional): Overlap between adjacent chunks. Defaults to 100.

    Returns:
        list[Document]: Chunked documents ready for embedding/indexing.

    Raises:
        ValueError: If inputs are invalid (empty docs or bad chunk parameters).
        RuntimeError: If chunking fails.
    """
    if not isinstance(documents, list) or not documents:
        raise ValueError("`documents` must be a non-empty list of Document objects.")
    if chunk_size <= 0:
        raise ValueError("`chunk_size` must be greater than 0.")
    if chunk_overlap < 0:
        raise ValueError("`chunk_overlap` must be 0 or greater.")
    if chunk_overlap >= chunk_size:
        raise ValueError("`chunk_overlap` must be smaller than `chunk_size`.")

    try:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        chunks = splitter.split_documents(documents)
    except Exception as exc:
        raise RuntimeError("Failed to split documents into chunks.") from exc

    if not chunks:
        raise ValueError("Chunking produced no output documents.")

    return chunks


def prepare_documents_from_hf(repo_id: str) -> list[Document]:
    """
    Load a dataset from Hugging Face and return chunked LangChain documents.

    Args:
        repo_id (str): Hugging Face dataset repository ID.

    Returns:
        list[Document]: Chunked LangChain documents ready for indexing.

    Raises:
        ValueError: If inputs are invalid or no documents/chunks are produced.
        RuntimeError: If loading or chunking fails.
    """
    docs = load_documents_from_hf(repo_id)
    return chunk_documents(docs)

In [ ]:
def is_vector_store_populated(collection_name: str = "medical_rag_v1") -> bool:
    """
    Check whether the persisted Chroma collection already contains vectors.

    Args:
        collection_name (str, optional): Name of the Chroma collection.

    Returns:
        bool: True if the collection exists and has at least one vector.
    """
    if not isinstance(collection_name, str) or not collection_name.strip():
        return False

    if not os.path.exists(DB_NAME):
        return False

    try:
        vector_store = Chroma(
            persist_directory=DB_NAME,
            embedding_function=embeddings,
            collection_name=collection_name,
        )
        return vector_store._collection.count() > 0
    except Exception as exc:
        print(f"Warning: unable to read existing vector store: {exc}")
        return False


def create_vector_store(
    documents: list[Document],
    collection_name: str = "medical_rag_v1",
    force_reingest: bool = False,
) -> Chroma:
    """
    Create and persist a Chroma vector store from chunked documents.

    Args:
        documents (list[Document]): Chunked medical documents.
        collection_name (str, optional): Name of the target Chroma collection.
        force_reingest (bool, optional): If True, clear existing collection before indexing.

    Returns:
        Chroma: A searchable vector database.

    Raises:
        ValueError: If inputs are invalid.
        RuntimeError: If collection deletion or indexing fails.
    """
    if not isinstance(documents, list) or not documents:
        raise ValueError("`documents` must be a non-empty list of chunked Document objects.")
    if not isinstance(collection_name, str) or not collection_name.strip():
        raise ValueError("`collection_name` must be a non-empty string.")

    print("🧠 Vectorizing medical knowledge...")

    if force_reingest and os.path.exists(DB_NAME):
        try:
            Chroma(
                persist_directory=DB_NAME,
                embedding_function=embeddings,
                collection_name=collection_name,
            ).delete_collection()
        except Exception as exc:
            raise RuntimeError(
                f"Failed to clear existing collection '{collection_name}' before re-ingest."
            ) from exc

    try:
        return Chroma.from_documents(
            documents=documents,
            embedding=embeddings,
            collection_name=collection_name,
            persist_directory=DB_NAME,
        )
    except Exception as exc:
        raise RuntimeError("Failed to create Chroma vector store from documents.") from exc


def get_or_create_vector_store(
    repo_id: str,
    collection_name: str = "medical_rag_v1",
    force_reingest: bool = False,
) -> Chroma:
    """
    Reuse an existing populated vector store or ingest from HF if missing.

    Args:
        repo_id (str): Hugging Face dataset repository ID.
        collection_name (str, optional): Name of the Chroma collection.
        force_reingest (bool, optional): If True, always rebuild from HF data.

    Returns:
        Chroma: Ready-to-query vector store.

    Raises:
        ValueError: If required inputs are invalid.
        RuntimeError: If reading/creating the vector store fails.
    """
    if not isinstance(repo_id, str) or not repo_id.strip():
        raise ValueError("`repo_id` must be a non-empty string.")
    if not isinstance(collection_name, str) or not collection_name.strip():
        raise ValueError("`collection_name` must be a non-empty string.")

    if not force_reingest and is_vector_store_populated(collection_name=collection_name):
        print("✅ Reusing existing vector store. Skipping HF ingestion.")
        try:
            return Chroma(
                persist_directory=DB_NAME,
                embedding_function=embeddings,
                collection_name=collection_name,
            )
        except Exception as exc:
            raise RuntimeError(
                "Detected existing vector store but failed to initialize it for retrieval."
            ) from exc

    print("📦 Building vector store from Hugging Face dataset...")
    all_docs = load_documents_from_hf(repo_id)
    chunked_docs = chunk_documents(all_docs)
    return create_vector_store(
        chunked_docs,
        collection_name=collection_name,
        force_reingest=force_reingest,
    )

In [ ]:
def build_medical_prompt() -> ChatPromptTemplate:
    """
    Build the RAG prompt used to answer medical questions.

    Returns:
        ChatPromptTemplate: Prompt with retrieved context and user input slots.

    Raises:
        RuntimeError: If prompt construction fails.
    """
    system_prompt = (
        "You are a medical assistant. Use the following pieces of retrieved "
        "medical dialogues to answer the user's question. \n\n"
        "{context}"
    )

    try:
        return ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                ("human", "{input}"),
            ]
        )
    except Exception as exc:
        raise RuntimeError("Failed to build medical prompt template.") from exc


def get_retriever(vector_store: Chroma, k: int = 4):
    """
    Create a similarity retriever from the vector store.

    Args:
        vector_store (Chroma): Backing vector database.
        k (int, optional): Number of chunks to retrieve. Defaults to 4.

    Returns:
        BaseRetriever: Configured retriever for RAG.

    Raises:
        ValueError: If retriever inputs are invalid.
        RuntimeError: If retriever creation fails.
    """
    if vector_store is None:
        raise ValueError("`vector_store` cannot be None.")
    if not isinstance(k, int) or k <= 0:
        raise ValueError("`k` must be a positive integer.")

    try:
        return vector_store.as_retriever(search_type="similarity", search_kwargs={"k": k})
    except Exception as exc:
        raise RuntimeError("Failed to create retriever from vector store.") from exc


def answer_question(vector_store: Chroma, question: str, k: int = 4) -> tuple[str, list[Document]]:
    """
    Answer a user question using retrieved medical context.

    Args:
        vector_store (Chroma): Vector store containing chunked medical dialogues.
        question (str): User question to answer.
        k (int, optional): Number of context chunks to retrieve. Defaults to 4.

    Returns:
        tuple[str, list[Document]]: Answer text and retrieved context documents.

    Raises:
        ValueError: If question or retrieval settings are invalid.
        RuntimeError: If the RAG chain fails to execute.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("`question` must be a non-empty string.")
    if not isinstance(k, int) or k <= 0:
        raise ValueError("`k` must be a positive integer.")

    try:
        prompt = build_medical_prompt()
        retriever = get_retriever(vector_store, k=k)

        combine_docs_chain = create_stuff_documents_chain(llm, prompt)
        retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

        response = retrieval_chain.invoke({"input": question})
    except Exception as exc:
        raise RuntimeError("Failed to generate an answer from the retrieval chain.") from exc

    if not isinstance(response, dict):
        raise RuntimeError("Retrieval chain returned an unexpected response format.")

    answer = response.get("answer", "")
    context_docs = response.get("context", [])

    if not isinstance(answer, str):
        answer = str(answer)
    if not isinstance(context_docs, list):
        context_docs = []

    return answer, context_docs

In [ ]:
# Example end-to-end usage + basic validation
repo_id = "jamalishaq/medical-qa-synthetic"
force_reingest = False  # Set to True to rebuild vectors from HF.

try:
    vector_store = get_or_create_vector_store(
        repo_id=repo_id,
        collection_name="medical_rag_v1",
        force_reingest=force_reingest,
    )

    question = "What should I do if I miss a dose of my blood pressure medication?"
    answer, retrieved_docs = answer_question(vector_store, question, k=4)

    print("\nAnswer:\n")
    print(answer)

    print("\nValidation:\n")
    print(f"Store populated: {is_vector_store_populated(collection_name='medical_rag_v1')}")
    print(f"Answer generated: {bool(answer.strip())}")
    print(f"Retrieved document count: {len(retrieved_docs)}")

    metadata_ok = all(
        "source_id" in doc.metadata and "niche" in doc.metadata
        for doc in retrieved_docs
    )
    print(f"Metadata contains source_id and niche: {metadata_ok}")

    print("\nRetrieved sources:\n")
    if not retrieved_docs:
        print("No context documents were retrieved.")
    else:
        for i, doc in enumerate(retrieved_docs, start=1):
            source_id = doc.metadata.get("source_id")
            niche = doc.metadata.get("niche")
            print(f"{i}. source_id={source_id}, niche={niche}")
except (ValueError, RuntimeError) as exc:
    print(f"Pipeline error: {exc}")

In [ ]:
import gradio as gr

UI_VECTOR_STORE = None
UI_DEFAULT_REPO_ID = "jamalishaq/medical-qa-synthetic"
UI_DEFAULT_COLLECTION = "medical_rag_v1"

In [ ]:
def get_ui_vector_store(
    repo_id: str = UI_DEFAULT_REPO_ID,
    collection_name: str = UI_DEFAULT_COLLECTION,
    force_reingest: bool = False,
):
    """Initialize and cache the vector store used by the Gradio UI."""
    global UI_VECTOR_STORE

    if UI_VECTOR_STORE is None or force_reingest:
        UI_VECTOR_STORE = get_or_create_vector_store(
            repo_id=repo_id,
            collection_name=collection_name,
            force_reingest=force_reingest,
        )

    return UI_VECTOR_STORE


def format_context(context_docs: list[Document], preview_chars: int = 280) -> str:
    """Render retrieved docs as markdown for the context panel."""
    if not context_docs:
        return "### Retrieved Context\nNo context documents were retrieved."

    sections = ["### Retrieved Context"]
    for i, doc in enumerate(context_docs, start=1):
        source_id = doc.metadata.get("source_id", "unknown")
        niche = doc.metadata.get("niche", "unknown")
        snippet = doc.page_content[:preview_chars].strip().replace("\n", " ")
        sections.append(f"**{i}. source_id={source_id}, niche={niche}**\n\n{snippet}...")

    return "\n\n".join(sections)


def chat_with_context(
    message: str,
    history: list[dict],
    vector_store: Chroma,
    retrieval_k: int = 4,
):
    """Handle one chat turn and return updated history + context markdown."""
    if not isinstance(history, list):
        history = []

    if not isinstance(message, str) or not message.strip():
        return "", history, "### Retrieved Context\nPlease enter a non-empty question."

    try:
        answer, context_docs = answer_question(vector_store, message, k=retrieval_k)
        assistant_text = answer if answer.strip() else "I could not generate an answer."
        context_md = format_context(context_docs)
    except Exception as exc:
        assistant_text = f"I hit an error while answering: {exc}"
        context_md = "### Retrieved Context\nNo context available due to an internal error."

    updated_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": assistant_text},
    ]

    return "", updated_history, context_md

In [ ]:
def launch_gradio_chat(
    repo_id: str = UI_DEFAULT_REPO_ID,
    collection_name: str = UI_DEFAULT_COLLECTION,
    retrieval_k: int = 4,
    force_reingest: bool = False,
):
    """Build and return a launchable Gradio chat interface for the medical RAG assistant."""
    vector_store = get_ui_vector_store(
        repo_id=repo_id,
        collection_name=collection_name,
        force_reingest=force_reingest,
    )

    def on_submit(message, history):
        return chat_with_context(message, history, vector_store, retrieval_k)

    theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

    with gr.Blocks(title="Medical RAG Assistant", theme=theme) as demo:
        gr.Markdown("# Medical RAG Assistant\nAsk medical questions grounded in retrieved dialogues.")

        with gr.Row():
            with gr.Column(scale=1):
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=560,
                    type="messages",
                    show_copy_button=True,
                )
                message = gr.Textbox(
                    label="Your Question",
                    placeholder="Ask a medical question...",
                    show_label=False,
                )

            with gr.Column(scale=1):
                context_markdown = gr.Markdown(
                    value="### Retrieved Context\nContext snippets will appear here.",
                    container=True,
                )

        message.submit(
            on_submit,
            inputs=[message, chatbot],
            outputs=[message, chatbot, context_markdown],
        )

    return demo

In [ ]:
# Launch from notebook
demo = launch_gradio_chat(
    repo_id=UI_DEFAULT_REPO_ID,
    collection_name=UI_DEFAULT_COLLECTION,
    retrieval_k=4,
    force_reingest=False,
)
demo.launch(inbrowser=True, share=False)